# 09 — Exposure measurement (Stage 6)

For each NE school, count salons within:

- **Euclidean buffers at 400 / 800 / 1600 m** (the conventional H2 comparator)
- **Walking-route buffers at 50 m and 100 m** (the novel measure)

Salon population: **manually verified** Google + OSM union (`confirmed` + `unsure` per `audit_logs/manual_verification.csv`). The verification filter is applied at the top of this notebook with an audit JSON written for the manuscript.

Output: `data/processed/exposure_panel_<date>.parquet` — one row per school, ready for Stage 7.

## 0. Pre-flight

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config, exposure, verification

config.ensure_dirs()
audit.verify_manifest()
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
TODAY

## 1. Load layers from earlier stages

In [ ]:
schools_path = sorted(
    p for p in config.DATA_PROCESSED.glob("schools_ne_*.gpkg") if "sensitivity" not in p.name
)[-1]
schools = gpd.read_file(schools_path)

google_path = sorted(config.DATA_PROCESSED.glob("salons_google_*.gpkg"))[-1]
google = gpd.read_file(google_path)
osm_path = sorted(config.DATA_PROCESSED.glob("salons_osm_*.gpkg"))[-1]
osm = gpd.read_file(osm_path)

school_bufs_path = sorted(config.DATA_PROCESSED.glob("school_euclidean_buffers_*.gpkg"))[-1]
school_bufs = gpd.read_file(school_bufs_path)
rb50_path = sorted(config.DATA_PROCESSED.glob("route_buffer_50m_*.gpkg"))[-1]
rb50 = gpd.read_file(rb50_path)
rb100_path = sorted(config.DATA_PROCESSED.glob("route_buffer_100m_*.gpkg"))[-1]
rb100 = gpd.read_file(rb100_path)

lsoa = gpd.read_file(sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))[-1])

print(f"Schools         : {len(schools):,}")
print(f"Google salons   : {len(google):,}")
print(f"OSM salons      : {len(osm):,}")
print(f"School buffers  : {len(school_bufs):,}")
print(f"Route buffers 50m: {len(rb50):,}")
print(f"Route buffers 100m: {len(rb100):,}")

## 2. Apply manual verification

Keep `confirmed` + `unsure` rows; reject `rejected` / `duplicate` / `closed`. Writes a per-run audit JSON to `audit_logs/`.

In [ ]:
ver = verification.load_verification(config.AUDIT_LOGS / "manual_verification.csv")
print("Verification status counts:")
print(verification.verification_summary(ver).to_string())

google_v, audit_g = verification.apply_verification(
    google, ver, source="google", source_id_col="place_id", audit_dir=config.AUDIT_LOGS
)
osm_v, audit_o = verification.apply_verification(
    osm, ver, source="osm", source_id_col="osm_id", audit_dir=config.AUDIT_LOGS
)
print(f"\nGoogle: {audit_g['n_kept']} of {audit_g['n_input']} kept")
print(f"OSM   : {audit_o['n_kept']} of {audit_o['n_input']} kept")

## 3. Build the verified-union salon layer

OSM rows whose `verification_id` was marked `duplicate` are already excluded by `apply_verification`. The remaining OSM-only rows are those that *aren't* in Google but ARE confirmed real salons. Union them with Google.

In [ ]:
g_min = google_v[["place_id", "name", "address", "lad_code", "imd_quintile", "geometry"]].copy()
g_min["source"] = "google"
g_min = g_min.rename(columns={"place_id": "unique_id"})

o_min = osm_v[["osm_id", "name", "addr_full", "lad_code", "imd_quintile", "geometry"]].copy()
o_min["source"] = "osm"
o_min = o_min.rename(columns={"osm_id": "unique_id", "addr_full": "address"})
o_min["unique_id"] = o_min["unique_id"].astype(str)
g_min["unique_id"] = g_min["unique_id"].astype(str)

salons = pd.concat([g_min, o_min], ignore_index=True)
salons = gpd.GeoDataFrame(salons, geometry="geometry", crs="EPSG:27700")
print(f"Verified union salon count: {len(salons)}")
print("Source split:")
print(salons['source'].value_counts().to_string())
print()
print("Per-LA verified salon counts:")
print(salons.groupby('lad_code').size().sort_values(ascending=False).to_string())

## 4. Buffer-based exposure (school-centred, 400 / 800 / 1600 m)

In [ ]:
import time

t0 = time.time()
buf_exposure = exposure.buffer_exposure(school_bufs, salons)
print(f"Buffer exposure computed in {time.time()-t0:.1f}s")
print(f"Schools with at least one salon in 1600 m: {(buf_exposure['n_buffer_1600m'] > 0).sum()}")
print(f"Schools with at least one salon in 400 m : {(buf_exposure['n_buffer_400m'] > 0).sum()}")
buf_exposure[["urn", "n_buffer_400m", "n_buffer_800m", "n_buffer_1600m"]].describe().round(1)

## 5. Route-based exposure (50 m primary, 100 m sensitivity)

In [ ]:
t0 = time.time()
rt50_exposure = exposure.route_exposure(rb50, salons, width_label="50m")
rt100_exposure = exposure.route_exposure(rb100, salons, width_label="100m")
print(f"Route exposure (50m + 100m) in {time.time()-t0:.1f}s")
print(f"\n50m primary stats:")
print(rt50_exposure[["sum_route_50m", "mean_per_pupil_route_50m", "max_route_50m", "n_routes_50m"]].describe().round(2))

## 6. Build the combined exposure panel

In [ ]:
panel = exposure.build_exposure_panel(
    schools, salons, school_bufs, rb50, rb100,
)
panel = panel.merge(
    schools[["urn", "n_pupils"]].rename(columns={"n_pupils": "n_pupils_school"}),
    on="urn", how="left", suffixes=("", "_dup")
)
panel = panel.loc[:, ~panel.columns.str.endswith("_dup")]
print(f"Panel: {len(panel)} schools × {len(panel.columns)} cols")
panel.head()

## 7. First-look H1 / H2 — exposure by IMD quintile

Pre-regression peek at the data: average exposure per school by IMD2025 quintile of the school's LSOA. If the route-based gradient is steeper than the buffer-based gradient, that's the H2 signal.

In [ ]:
schools_imd = schools.sjoin(
    lsoa[["lsoa21cd", "imd_quintile", "geometry"]], how="left", predicate="within"
)[["urn", "imd_quintile"]]
panel_imd = panel.merge(schools_imd, on="urn", how="left")

by_q = panel_imd.groupby("imd_quintile").agg(
    n_schools=("urn", "count"),
    mean_n_buffer_800m=("n_buffer_800m", "mean"),
    mean_per_pupil_route_50m=("mean_per_pupil_route_50m", "mean"),
).round(2)
by_q

In [ ]:
import numpy as np

print("Buffer 800 m gradient (Q1 most deprived / Q5 least deprived):")
ratio_b = by_q.loc[1, "mean_n_buffer_800m"] / max(by_q.loc[5, "mean_n_buffer_800m"], 1e-6)
print(f"  Q1 / Q5 = {ratio_b:.2f}")
print(f"\nRoute 50 m mean-per-pupil gradient (Q1 / Q5):")
ratio_r = by_q.loc[1, "mean_per_pupil_route_50m"] / max(by_q.loc[5, "mean_per_pupil_route_50m"], 1e-6)
print(f"  Q1 / Q5 = {ratio_r:.2f}")
print(f"\nH2-style ratio amplification: route / buffer = {ratio_r / ratio_b:.2f}")
print("  (>1 means walking-route exposure shows a steeper gradient than school-centred buffer)")

## 8. Persist outputs

In [ ]:
out_panel = config.DATA_PROCESSED / f"exposure_panel_{TODAY}.parquet"
out_salons = config.DATA_PROCESSED / f"salons_verified_union_{TODAY}.gpkg"
panel_imd.to_parquet(out_panel, index=False)
salons.to_file(out_salons, driver="GPKG", layer="salons_verified_union")

for path, inputs, notes in (
    (
        out_panel,
        (school_bufs_path, rb50_path, rb100_path, google_path, osm_path, schools_path),
        "Stage 6 exposure panel: per-school buffer + route counts (verified salons only).",
    ),
    (
        out_salons,
        (google_path, osm_path, config.AUDIT_LOGS / "manual_verification.csv"),
        "Stage 6 verified-union salon layer (confirmed + unsure).",
    ),
):
    audit.write_provenance_sidecar(
        audit.Provenance(output_path=path, inputs=inputs, notes=notes, random_seed=config.RANDOM_SEED)
    )
    print("wrote:", path.name)

## Done

Outputs:
- `data/processed/exposure_panel_<date>.parquet` (per-school exposure measures + IMD)
- `data/processed/salons_verified_union_<date>.gpkg` (the analytic salon cohort)

Stage 7 (`11_primary_models_h1_h2_h3.ipynb`) reads the panel and runs the negative-binomial regressions for H1 / H2 / H3.